# GRPO Training for DAG Planner

Fine-tune a 7B model to generate better DAG plans from the tool catalogue.
Uses scored data from `train_planner.py` (collected locally) as the reward signal.

**Setup:** Kaggle notebook with T4 GPU accelerator.

**Approach:** Offline GRPO — rewards were pre-computed by executing DAGs locally.
We train the model to produce outputs that scored reward >= 1.0 (exact match)
and avoid outputs that scored -0.5 or -1.0.

In [ ]:
!pip install -q trl peft bitsandbytes accelerate transformers datasets

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## 1. Load scored data

Upload `data/planner_scores.jsonl` to Kaggle as a dataset, or put it in the notebook's input.
We filter to only the **winning** examples (reward >= 0.5) as positive training signal.

In [ ]:
# -- Configuration --
SCORES_PATH = "/kaggle/input/planner-scores/planner_scores.jsonl"  # adjust to your upload path
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "/kaggle/working/planner-lora"
MIN_REWARD = 0.5  # only train on examples that got the answer right

In [ ]:
PLANNER_SYSTEM = """\
You are a unified DAG planner for reasoning puzzles.  You receive a
PUZZLE_TYPE and a PROMPT, and output an execution plan.

OUTPUT FORMAT (strictly two sections, nothing else):

MERMAID:
START --> N1
N1 --> N2
...
Nk --> END

NODES:
{"N1": {"id": "...", "question": "...", "tool": "...", "tool_input": "..."}, ...}

FIELD RULES:
- id        : unique snake_case identifier for the node.
- question  : one-sentence description of what the node does.
- tool      : exact tool function name from the catalogue below.
- tool_input: a JSON *string*.  Use \"__PROMPT__\" (with quotes) wherever
              the full puzzle prompt text is needed — it will be replaced
              at runtime.  Use {parent_id} for dependency interpolation
              and {parent_id_field} for JSON sub-field access.
- depends_on is NOT in NODES — it is derived from the MERMAID edges.

TOOL CATALOGUE AND PLANS PER PUZZLE TYPE:

=== gravity_physics (3 nodes) ===
N1  id=extract_obs     tool=extract_gravity_obs
    tool_input = {\"prompt\": \"__PROMPT__\"}
N2  id=compute_g       tool=compute_gravity_g       depends on N1
    tool_input = {extract_obs}
N3  id=predict_d       tool=compute_gravity_d       depends on N1, N2
    tool_input = {\"g\": \"{compute_g}\", \"t\": \"{extract_obs_target_t}\"}

=== unit_conversion (3 nodes) ===
N1  id=extract_pairs   tool=extract_unit_pairs
    tool_input = {\"prompt\": \"__PROMPT__\"}
N2  id=compute_factor  tool=geometric_mean_factor   depends on N1
    tool_input = {extract_pairs}
N3  id=predict         tool=apply_factor_round      depends on N1, N2
    tool_input = {\"factor\": \"{compute_factor}\", \"target\": \"{extract_pairs_target}\"}

=== numeral_conversion (1 node) ===
N1  id=solve_numeral   tool=solve_numeral_conversion
    tool_input = {\"prompt\": \"__PROMPT__\"}

=== cipher_decryption (2*N + 2 nodes) ===
For EACH example line i (i = 1..N) in the prompt:
  N(2i-1) id=extract_pairs_i   tool=split_word_pairs     root node
          tool_input = {\"encrypted\": \"<line i encrypted text>\", \"plaintext\": \"<line i plain text>\"}
  N(2i)   id=create_mapping_i  tool=build_char_map       depends on extract_pairs_i
          tool_input = {extract_pairs_i}
Then two final nodes:
  merge_mapping   tool=merge_char_maps        depends on ALL create_mapping_i
          tool_input = newline-joined {create_mapping_i} refs
  translate       tool=decrypt_substitution   depends on merge_mapping
          tool_input = {\"ciphertext\": \"<target ciphertext>\", \"mapping\": {merge_mapping}}
Read the actual encrypted/plain/ciphertext from the PROMPT.

=== bit_manipulation (1 node) ===
N1  id=solve_bits      tool=solve_bit_manipulation
    tool_input = {\"prompt\": \"__PROMPT__\"}

=== equation_transform (1 node) ===
N1  id=solve_equation  tool=solve_equation_transform
    tool_input = {\"prompt\": \"__PROMPT__\"}

RULES:
- Match the plan to PUZZLE_TYPE exactly.
- Use the EXACT tool names and node IDs shown above.
- Output ONLY MERMAID and NODES sections — no commentary.\
"""

print(f"System prompt: {len(PLANNER_SYSTEM)} chars")

In [ ]:
# Load and filter scored data
raw_lines = [json.loads(l) for l in open(SCORES_PATH, encoding="utf-8")]
print(f"Total samples: {len(raw_lines)}")

# Keep only winning examples as SFT targets
winners = [r for r in raw_lines if r["reward"] >= MIN_REWARD and r["dag_valid"]]
losers = [r for r in raw_lines if r["reward"] < 0 and r["dag_valid"]]
print(f"Winners (reward >= {MIN_REWARD}): {len(winners)}")
print(f"Losers  (reward < 0, valid DAG): {len(losers)}")

# Distribution by type
from collections import Counter
print("\nWinners by type:", Counter(r["puzzle_type"] for r in winners))
print("Losers by type:", Counter(r["puzzle_type"] for r in losers))

## 2. Build training dataset

We format each winning example as a chat conversation:
- **system**: PLANNER_SYSTEM
- **user**: PUZZLE_TYPE + PROMPT (reconstructed from the scored data)
- **assistant**: the planner_output that scored well

This is supervised fine-tuning (SFT) on the best plans — the simplest
effective approach. For full GRPO you'd need online generation, which
requires the tool execution pipeline on Kaggle.

In [ ]:
def build_chat(record):
    """Convert a scored record into a chat-format training example."""
    user_msg = f"PUZZLE_TYPE: {record['puzzle_type']}\n\nPROMPT:\n{record['prompt']}"
    return {
        "messages": [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": record["planner_output"]},
        ]
    }

# Deduplicate: keep best candidate per puzzle_id
best_per_puzzle = {}
for r in winners:
    pid = r['puzzle_id']
    if pid not in best_per_puzzle or r['reward'] > best_per_puzzle[pid]['reward']:
        best_per_puzzle[pid] = r

train_data = [build_chat(r) for r in best_per_puzzle.values()]
dataset = Dataset.from_list(train_data)
print(f"Training examples: {len(dataset)} (deduplicated from {len(winners)} winners)")
print(f"Sample:\n{json.dumps(train_data[0]['messages'][-1]['content'][:300], indent=2)}")

## 3. Load base model (4-bit quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"Model loaded: {BASE_MODEL}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 4. Apply LoRA adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Train

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    max_seq_length=2048,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

## 6. Save LoRA adapter

Download the adapter weights from Kaggle output. They're small (~50-100MB)
and can be loaded on top of the base model locally or via OpenRouter.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

import os
total_size = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in os.listdir(OUTPUT_DIR))
print(f"Total adapter size: {total_size/1e6:.1f} MB")

## 7. Quick sanity check — generate a plan

In [ ]:
test_prompt = """PUZZLE_TYPE: gravity_physics

PROMPT:
In Alice's Wonderland, the gravitational constant has been secretly changed. Here are some example observations:
For t = 1.37s, distance = 14.92 m
For t = 4.27s, distance = 144.96 m
For t = 3.28s, distance = 85.54 m
Now, determine the falling distance for t = 4.41s given d = 0.5*g*t^2."""

messages = [
    {"role": "system", "content": PLANNER_SYSTEM},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
with torch.no_grad():
    output = model.generate(inputs, max_new_tokens=512, temperature=0.3, do_sample=True)

generated = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
print("Generated plan:")
print(generated)